# FreeFine Final Geometry — Step 1: Deployable Router Preflight (balanced-200)

This is a **cheap preflight only**, not the final benchmark evaluation. It uses the already-generated balanced-200 outputs to verify that the newly frozen **exact-affine** resize rule behaves sensibly before spending GPU quota on the full 5,677 end-to-end run.

Important methodological point: the earlier `resize_hard` gains were measured on the thesis diagnostic tertile. Step 0 v2 showed that this diagnostic label is not identical to the deployable affine severity rule. Therefore we must evaluate the **actual final routing rule once** before the expensive full run.

No image generation is performed here. The notebook reconstructs the two proposed pipelines from already-generated outputs and runs the official 7 metrics on the standard balanced-200 development set.

**Pipeline A — SGR-EPSREC**
- move → RING4_MOVE_POST
- rotate → original FreeFine
- resize non-severe → RING8_GLOBAL_POST
- resize severe (`scale >= 1.5` or `scale <= 0.6`) → EPSREC_PROMPT_ALL

**Pipeline B — SGR-MIDHF+EPSREC**
- identical except severe resize → MIDHF_EPSREC_PROMPT_ALL


In [1]:
import os,time,subprocess,json,glob,shutil,csv,re,hashlib,random
NB_START=time.time()
COMMIT="4c9fdb971572b32edbeac13464659274c28decbb"
subprocess.run(
    f"mkdir -p /kaggle/temp && cd /kaggle/temp && rm -rf FreeFine && "
    f"git clone -q https://github.com/CIawevy/FreeFine.git && "
    f"cd FreeFine && git checkout -q {COMMIT}",
    shell=True,check=True
)
print("✓ cloned + pinned FreeFine",COMMIT[:14])


✓ cloned + pinned FreeFine 4c9fdb971572b3


In [2]:
%%bash
set -e
V=/kaggle/temp/metric_env
PY=$V/bin/python
REPO=/kaggle/temp/FreeFine
rm -rf "$V"
uv venv --python 3.10.13 "$V"
uv pip install --python "$PY" torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
uv pip install --python "$PY" "setuptools<70" wheel pip
grep -vi '^clip' $REPO/evaluation/metrics/requirements.txt > /tmp/m.txt
uv pip install --python "$PY" -r /tmp/m.txt
uv pip install --python "$PY" "setuptools<70"
uv pip install --python "$PY" --no-build-isolation "clip @ git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1"
uv pip install --python "$PY" "pyarrow<16" "datasets<3"
wget -q https://dl.fbaipublicfiles.com/mmf/clip/bpe_simple_vocab_16e6.txt.gz -P /tmp
for d in $(find $V -path '*/site-packages/clip' -o -path '*open_clip' -type d); do
  cp /tmp/bpe_simple_vocab_16e6.txt.gz "$d/" 2>/dev/null || true
done
echo "✓ metric_env OK"


✓ metric_env OK


 Downloaded cpython-3.10.13-linux-x86_64-gnu (download)
Using CPython 3.10.13
Creating virtual environment at: /kaggle/temp/metric_env
Activate with: source /kaggle/temp/metric_env/bin/activate
Using Python 3.10.13 environment at: /kaggle/temp/metric_env
Resolved 27 packages in 346ms
 Downloaded torchaudio
 Downloaded nvidia-cuda-cupti-cu12
 Downloaded nvidia-cuda-nvrtc-cu12
 Downloaded torchvision
 Downloaded nvidia-nvjitlink-cu12
 Downloaded pillow
 Downloaded nvidia-cufft-cu12
 Downloaded networkx
 Downloaded nvidia-curand-cu12
 Downloaded triton
 Downloaded nvidia-nccl-cu12
 Downloaded nvidia-cudnn-cu12
 Downloaded numpy
 Downloaded nvidia-cusolver-cu12
 Downloaded nvidia-cusparse-cu12
 Downloaded nvidia-cusparselt-cu12
 Downloaded sympy
 Downloaded nvidia-cublas-cu12
 Downloaded torch
Prepared 27 packages in 31.41s
Installed 27 packages in 413ms
 + filelock==3.32.3
 + fsspec==2026.7.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + nvid

In [3]:
import pathlib,re,py_compile
mr=pathlib.Path("/kaggle/temp/FreeFine/evaluation/metrics")
mp=mr/"main.py"
s=mp.read_text()
s=s.replace("args.3d","getattr(args,'3d')")
anchor="    args = parser.parse_args()\n"
assert anchor in s
seed_patch=anchor+"""    import random as _random, numpy as _np
    try:
        import torch as _torch
    except Exception:
        _torch=None
    _metric_seed=int(os.environ.get('FF_METRIC_SEED','42'))
    _random.seed(_metric_seed); _np.random.seed(_metric_seed)
    if _torch is not None:
        _torch.manual_seed(_metric_seed)
        if _torch.cuda.is_available(): _torch.cuda.manual_seed_all(_metric_seed)
"""
s=s.replace(anchor,seed_patch,1)
mp.write_text(s)

for f in [mr/"MD"/"mean_distance.py",mr/"MD"/"dift_sd.py"]:
    f.write_text(f.read_text().replace(
        "stabilityai/stable-diffusion-2-1",
        "sd2-community/stable-diffusion-2-1"
    ))
md=mr/"MD"/"mean_distance.py"
s=md.read_text()
needle="all_dist = []"
assert needle in s
s=s.replace(
    needle,
    needle+"\n    import torch as _st, os as _os; "
           "_seed=int(_os.environ.get('FF_MD_SEED','42')); "
           "_st.manual_seed(_seed); _st.cuda.manual_seed_all(_seed)",
    1
)
md.write_text(s)
for f in [mr/"main.py",mr/"MD"/"mean_distance.py",mr/"MD"/"dift_sd.py"]:
    py_compile.compile(str(f),doraise=True)
print("✓ deterministic metric patches installed")


✓ deterministic metric patches installed


In [4]:
import os,glob,json,csv,random,shutil,hashlib,math
from collections import defaultdict,Counter
import numpy as np
import pandas as pd

GEO="/kaggle/temp/GeoBenchMeta"
os.makedirs(f"{GEO}/Geo-Bench-2D",exist_ok=True)

def one(pattern,desc):
    xs=glob.glob(pattern,recursive=True)
    if not xs:
        raise FileNotFoundError(
            f"Missing {desc}. Attach the same GeoBench + baseline inputs used by Wave 1/2. Pattern={pattern}"
        )
    return xs[0]

cc=[c for c in glob.glob("/kaggle/input/**/Geo-Bench-2D",recursive=True)
    if os.path.isdir(f"{c}/source_img")]
if not cc:
    raise FileNotFoundError("Geo-Bench-2D/source_img not found under /kaggle/input")
CACHE=cc[0]
COARSE=one("/kaggle/input/**/coarse_img/*/*/*.png","coarse_img").split("/coarse_img/")[0]+"/coarse_img"
GENBASE=one("/kaggle/input/**/gen_results_2d_final/gen_results_2d_backup","baseline generation directory")
IB=os.path.dirname(os.path.dirname(os.path.dirname(
    one("/kaggle/input/**/inp_img_blended/**/inp_img.png","inp_img_blended")
)))
ANNs=one("/kaggle/input/**/annotation_2d.json","annotation_2d.json")
META=one("/kaggle/input/**/sample_metadata.csv","sample_metadata.csv")

for nm in ["source_img","source_mask","target_mask","source_img_full_v2"]:
    d=f"{GEO}/Geo-Bench-2D/{nm}"
    if os.path.lexists(d):
        os.remove(d) if os.path.islink(d) else shutil.rmtree(d)
    os.symlink(f"{CACHE}/{nm}",d)
for nm,sc in [("coarse_img",COARSE),("inp_img_blended",IB)]:
    d=f"{GEO}/Geo-Bench-2D/{nm}"
    if os.path.lexists(d):
        os.remove(d) if os.path.islink(d) else shutil.rmtree(d)
    os.symlink(sc,d)

shutil.copy(ANNs,f"{GEO}/annotation_2d.json")
ann=json.load(open(f"{GEO}/annotation_2d.json"))
meta=[r for r in csv.DictReader(open(META))
      if os.path.exists(f"{IB}/{r['da_n']}/{r['ins_id']}/inp_img.png")]

# Exact historical balanced-200 reconstruction.
random.seed(42)
bycell=defaultdict(list)
for r in meta:
    bycell[(r["edit_type"],r["difficulty"])].append(r)
keys=sorted(bycell)
per=200//len(keys)
picked=[]
for k in keys:
    pool=bycell[k][:]
    random.shuffle(pool)
    picked += pool[:per]
chosen={(r["da_n"],r["ins_id"],r["case_id"]) for r in picked}
left=[r for r in meta if (r["da_n"],r["ins_id"],r["case_id"]) not in chosen]
random.shuffle(left)
for r in left:
    if len(picked)>=200: break
    picked.append(r)
picked=picked[:200]

sha=hashlib.sha256(
    "\n".join(sorted(f"{r['da_n']}|{r['ins_id']}|{r['case_id']}" for r in picked)).encode()
).hexdigest()
assert sha=="3d7c0172cba1e35693a3f20b562004bcfd6fc43e187160c945a23224d40cbd3c",sha
print("✓ exact balanced-200",Counter(r["edit_type"] for r in picked),sha)

# Attach exact affine scale to each row from the inference annotation.
for r in picked:
    ep=ann[str(r["da_n"])]["instances"][str(r["ins_id"])][str(r["case_id"])]["edit_param"]
    assert len(ep)==9
    sx=float(ep[6]); sy=float(ep[7])
    r["sx_exact"]=sx
    r["sy_exact"]=sy
    r["scale_exact"]=math.sqrt(sx*sy)
    r["affine_severe"]=(r["edit_type"]=="resize" and (r["scale_exact"]>=1.5 or r["scale_exact"]<=0.6))

res=[r for r in picked if r["edit_type"]=="resize"]
print("resize 200-slice n=",len(res))
print("affine severe=",sum(r["affine_severe"] for r in res),
      "nonsevere=",sum(not r["affine_severe"] for r in res))
print("diagnostic hard x affine severe:")
print(pd.crosstab(
    pd.Series([r["difficulty"]=="hard" for r in res],name="diagnostic_hard"),
    pd.Series([r["affine_severe"] for r in res],name="affine_severe")
))

json.dump(picked,open(f"{GEO}/subset_meta_affine_router.json","w"),indent=2)

EVALROOT=f"{GEO}/gen_eval"
os.makedirs(EVALROOT,exist_ok=True)
base_link=f"{EVALROOT}/baseline"
if os.path.lexists(base_link):
    os.remove(base_link) if os.path.islink(base_link) else shutil.rmtree(base_link)
os.symlink(GENBASE,base_link)

def key(r): return (r["da_n"],r["ins_id"],r["case_id"])
def mem(pred): return [key(r) for r in picked if pred(r)]

groups={
    "all_200":mem(lambda r:True),
    "move_all":mem(lambda r:r["edit_type"]=="move"),
    "rotate_all":mem(lambda r:r["edit_type"]=="rotate"),
    "resize_all":mem(lambda r:r["edit_type"]=="resize"),
    "resize_affine_severe":mem(lambda r:r["edit_type"]=="resize" and r["affine_severe"]),
    "resize_affine_nonsevere":mem(lambda r:r["edit_type"]=="resize" and not r["affine_severe"]),
}
print("group sizes:",{k:len(v) for k,v in groups.items()})


✓ exact balanced-200 Counter({'move': 67, 'resize': 67, 'rotate': 66}) 3d7c0172cba1e35693a3f20b562004bcfd6fc43e187160c945a23224d40cbd3c
resize 200-slice n= 67
affine severe= 20 nonsevere= 47
diagnostic hard x affine severe:
affine_severe    False  True 
diagnostic_hard              
False               41      3
True                 6     17
group sizes: {'all_200': 200, 'move_all': 67, 'rotate_all': 66, 'resize_all': 67, 'resize_affine_severe': 20, 'resize_affine_nonsevere': 47}


In [5]:
import os,glob,json,shutil,numpy as np,cv2
from PIL import Image

RAW=["EPSREC_PROMPT_ALL","MIDHF_EPSREC_PROMPT_ALL"]
roots={}

def pc(p): return len(glob.glob(p+"/**/*.png",recursive=True)) if os.path.isdir(p) else 0

for tag in RAW:
    cands=glob.glob(f"/kaggle/input/**/finalB_guidance_move/variants/{tag}",recursive=True)
    if not cands:
        raise FileNotFoundError(
            f"Missing raw output {tag}. Attach original Account-B-v3 saved output."
        )
    root=max(cands,key=pc)
    n=pc(root)
    assert n>=200,(tag,n,root)
    roots[tag]=root
    print(tag,n,root)

BASE=f"{EVALROOT}/baseline"
WORK="/kaggle/working/final_geometry_preflight"
os.makedirs(WORK,exist_ok=True)

def reset(p):
    if os.path.lexists(p):
        os.remove(p) if os.path.islink(p) or os.path.isfile(p) else shutil.rmtree(p)
    os.makedirs(p,exist_ok=True)

def link_eval(tag,root):
    d=f"{EVALROOT}/{tag}"
    if os.path.lexists(d):
        os.remove(d) if os.path.islink(d) else shutil.rmtree(d)
    os.symlink(root,d)

# Build exact post variants from the same baseline images used in the previous experiments.
def ring_image(bp,cp,tp,w):
    G=np.array(Image.open(bp).convert("RGB"))
    C=np.array(Image.open(cp).convert("RGB"))
    T=np.array(Image.open(tp).convert("L"))>127
    interior=cv2.erode(
        T.astype(np.uint8),
        np.ones((2*w+1,2*w+1),np.uint8),
        iterations=1
    ).astype(bool)
    return np.where(interior[:,:,None],C,G).astype(np.uint8)

R4=f"{WORK}/RING4_MOVE"
R8=f"{WORK}/RING8_RESIZE"
reset(R4); reset(R8)

for r in picked:
    d,i,e=key(r)
    bp=f"{BASE}/{d}/{i}/{e}.png"
    assert os.path.exists(bp),bp
    cp=f"{GEO}/Geo-Bench-2D/coarse_img/{d}/{i}/{e}.png"
    tp=f"{GEO}/Geo-Bench-2D/target_mask/{d}/{i}/{e}.png"

    if r["edit_type"]=="move":
        dst=f"{R4}/{d}/{i}/{e}.png"; os.makedirs(os.path.dirname(dst),exist_ok=True)
        Image.fromarray(ring_image(bp,cp,tp,4)).save(dst)
    if r["edit_type"]=="resize":
        dst=f"{R8}/{d}/{i}/{e}.png"; os.makedirs(os.path.dirname(dst),exist_ok=True)
        Image.fromarray(ring_image(bp,cp,tp,8)).save(dst)

def build_pipeline(tag,severe_source):
    root=f"{WORK}/{tag}"
    reset(root)
    for r in picked:
        d,i,e=key(r)
        if r["edit_type"]=="move":
            src=f"{R4}/{d}/{i}/{e}.png"
        elif r["edit_type"]=="rotate":
            src=f"{BASE}/{d}/{i}/{e}.png"
        elif r["affine_severe"]:
            src=f"{roots[severe_source]}/{d}/{i}/{e}.png"
        else:
            src=f"{R8}/{d}/{i}/{e}.png"
        assert os.path.exists(src),src
        dst=f"{root}/{d}/{i}/{e}.png"
        os.makedirs(os.path.dirname(dst),exist_ok=True)
        # copy, not symlink, so saved outputs are self-contained
        shutil.copy2(src,dst)
    assert pc(root)==200,(tag,pc(root))
    link_eval(tag,root)
    print("built",tag,"200/200")
    return root

A=build_pipeline("SGR_EPSREC_AFFINE","EPSREC_PROMPT_ALL")
B=build_pipeline("SGR_MIDHF_EPSREC_AFFINE","MIDHF_EPSREC_PROMPT_ALL")
print("✓ both deployable-router preflight composites built")


EPSREC_PROMPT_ALL 200 /kaggle/input/notebooks/giorgostzam/freefine-final-exhaustive-run-account-b-guidan/finalB_guidance_move/variants/EPSREC_PROMPT_ALL
MIDHF_EPSREC_PROMPT_ALL 200 /kaggle/input/notebooks/giorgostzam/freefine-final-exhaustive-run-account-b-guidan/finalB_guidance_move/variants/MIDHF_EPSREC_PROMPT_ALL
built SGR_EPSREC_AFFINE 200/200
built SGR_MIDHF_EPSREC_AFFINE 200/200
✓ both deployable-router preflight composites built


In [6]:
import os,json,re,subprocess,time,threading,queue,pandas as pd
MET="/kaggle/temp/FreeFine/evaluation/metrics"
PY="/kaggle/temp/metric_env/bin/python"

def manifest(tag,group_name,ids):
    out={}; base=f"{EVALROOT}/{tag}"; used=0
    for d,i,e in ids:
        gp=f"{base}/{d}/{i}/{e}.png"
        if not os.path.exists(gp): continue
        lf=dict(ann[d]["instances"][i][e])
        lf["gen_img_path"]=f"gen_eval/{tag}/{d}/{i}/{e}.png"
        out.setdefault(d,{"instances":{}})["instances"].setdefault(i,{})[e]=lf
        used+=1
    p=f"{GEO}/m_{re.sub(r'[^A-Za-z0-9_.-]+','_',tag+'_'+group_name)}.json"
    json.dump(out,open(p,"w"))
    return p,used

def run_metric(manifest_path,task,gpu,md_seed=42,metric_seed=42):
    env=os.environ.copy()
    env.update({
        "CUDA_VISIBLE_DEVICES":str(gpu),
        "MPLBACKEND":"Agg",
        "HF_HOME":"/kaggle/temp/hf",
        "TORCH_HOME":"/kaggle/temp/torch",
        "PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True",
        "FF_MD_SEED":str(md_seed),
        "FF_METRIC_SEED":str(metric_seed),
        "TOKENIZERS_PARALLELISM":"false",
    })
    p=subprocess.run(
        [PY,"main.py","--path",manifest_path,"--use_relative_path","--base_dir",GEO,
         "--fid_path",f"{GEO}/Geo-Bench-2D/source_img_full_v2",
         "--task",task,"--level","0"],
        cwd=MET,env=env,capture_output=True,text=True
    )
    txt=p.stdout+p.stderr
    vals={}
    for k in ["FID_DINO","FID_KD","FID","SUBC","BGC","WRAP_E","MD"]:
        h=re.findall(rf"(?:^|\s){k}:\s*([-\d.eE]+)",txt)
        if h: vals[k]=round(float(h[-1]),4)
    if p.returncode!=0:
        vals["_rc"]=p.returncode; vals["_tail"]=txt[-1800:]
    return vals

def parallel_jobs(jobs,worker_fn,deadline_hours=11.6):
    q=queue.Queue()
    for x in jobs:q.put(x)
    deadline=NB_START+deadline_hours*3600
    def _w(gpu):
        while time.time()<deadline-120:
            try:job=q.get_nowait()
            except queue.Empty:return
            try:worker_fn(gpu,*job)
            finally:q.task_done()
    ts=[threading.Thread(target=_w,args=(g,),daemon=True) for g in [0,1]]
    [t.start() for t in ts]; [t.join() for t in ts]
    return list(q.queue)
print("✓ metric helpers ready")


✓ metric helpers ready


In [7]:
# Evaluate baseline and both exact-affine final proposals.
TAGS=["baseline","SGR_EPSREC_AFFINE","SGR_MIDHF_EPSREC_AFFINE"]
CORE_GROUPS=["all_200","move_all","rotate_all","resize_all","resize_affine_severe","resize_affine_nonsevere"]
FID_GROUPS=["all_200","move_all","rotate_all","resize_all"]

RESULT_JSON="/kaggle/working/final_geometry_router_preflight_results.json"
RESULT_CSV="/kaggle/working/final_geometry_router_preflight_results.csv"
results={}
lock=threading.Lock()

# Canonical baseline gate before doing anything else.
m,u=manifest("baseline","all_200",groups["all_200"])
gate=run_metric(m,"000111100",0,42,42)
assert u==200
assert abs(gate.get("SUBC",-9)-0.9154)<5e-4,gate
assert abs(gate.get("BGC",-9)-0.9657)<8e-4,gate
assert abs(gate.get("WRAP_E",-9)-0.0488)<8e-4,gate
assert abs(gate.get("MD",-9)-7.6023)<0.03,gate
print("✓ baseline core gate",gate)

def save():
    with lock:
        json.dump(results,open(RESULT_JSON,"w"),indent=2)
        rows=[]
        for tag,gd in results.items():
            for g,v in gd.items():
                rows.append({"set":tag,"group":g,**v})
        pd.DataFrame(rows).to_csv(RESULT_CSV,index=False)

jobs=[]
for tag in TAGS:
    for g in CORE_GROUPS:
        jobs.append((tag,g,"core"))
    for g in FID_GROUPS:
        jobs.append((tag,g,"fid"))

def work(gpu,tag,g,kind):
    m,u=manifest(tag,g,groups[g])
    if u!=len(groups[g]):
        raise RuntimeError((tag,g,u,len(groups[g])))
    task="000111100" if kind=="core" else "100000011"
    v=run_metric(m,task,gpu,42,42)
    with lock:
        old=results.setdefault(tag,{}).setdefault(g,{})
        old.update(v)
        old["n"]=u
        old["expected_n"]=len(groups[g])
        old["complete_group"]=True
    save()
    print(f"[GPU{gpu}] {tag:28s} {g:24s} {kind:4s} -> {v}",flush=True)

remaining=parallel_jobs(jobs,work,11.65)
save()
assert not remaining,f"unfinished metric jobs: {remaining}"

# Completeness
required_core=["SUBC","BGC","WRAP_E","MD"]
required_fid=["FID","FID_DINO","FID_KD"]
missing=[]
for tag in TAGS:
    for g in CORE_GROUPS:
        v=results.get(tag,{}).get(g,{})
        if any(k not in v for k in required_core): missing.append((tag,g,"core"))
    for g in FID_GROUPS:
        v=results.get(tag,{}).get(g,{})
        if any(k not in v for k in required_fid): missing.append((tag,g,"fid"))
print("MISSING",len(missing),missing)
assert not missing

# Compact comparison
metrics=["SUBC","BGC","WRAP_E","MD","FID","FID_DINO","FID_KD"]
for g in ["all_200","move_all","rotate_all","resize_all","resize_affine_severe","resize_affine_nonsevere"]:
    print("\n===",g,"===")
    for tag in TAGS:
        v=results[tag][g]
        print(tag,{k:v.get(k) for k in metrics if k in v})

print("\n✓ STEP 1 PREFLIGHT COMPLETE")
print("saved",RESULT_JSON)
print("saved",RESULT_CSV)


✓ baseline core gate {'SUBC': 0.9154, 'BGC': 0.9657, 'WRAP_E': 0.0488, 'MD': 7.6023}
[GPU1] baseline                     move_all                 core -> {'SUBC': 0.959, 'BGC': 0.9639, 'WRAP_E': 0.0495, 'MD': 3.408}
[GPU1] baseline                     rotate_all               core -> {'SUBC': 0.8962, 'BGC': 0.9684, 'WRAP_E': 0.0423, 'MD': 9.6294}
[GPU0] baseline                     all_200                  core -> {'SUBC': 0.9154, 'BGC': 0.9657, 'WRAP_E': 0.0488, 'MD': 7.6023}
[GPU0] baseline                     resize_affine_severe     core -> {'SUBC': 0.8185, 'BGC': 0.9661, 'WRAP_E': 0.0573, 'MD': 22.0809}
[GPU1] baseline                     resize_all               core -> {'SUBC': 0.8906, 'BGC': 0.9647, 'WRAP_E': 0.0544, 'MD': 10.9331}
[GPU1] baseline                     all_200                  fid  -> {'FID_DINO': 1636.8248, 'FID_KD': 0.1268, 'FID': 132.279}
[GPU1] baseline                     move_all                 fid  -> {'FID_DINO': 2444.5409, 'FID_KD': 0.0259, 'FID': 200.0